In [1]:
import re

import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("../data/movies_enriched.csv")
tags = pd.read_csv("../data/ml-latest-small/tags.csv")

print(df.shape)

(9742, 16)


In [2]:
def limpar_token(texto):
    """'Sci-Fi' -> 'scifi', 'Christopher Nolan' -> 'christophernolan'."""
    return re.sub(r"[^a-z0-9]", "", texto.lower())


def lista_para_tokens(valor, limite=None):
    """Transforma 'A B|C D' em 'ab cd', um token por item."""
    if pd.isna(valor) or valor == "":
        return ""
    itens = valor.split("|")
    if limite:
        itens = itens[:limite]
    tokens = [limpar_token(i) for i in itens]
    return " ".join(t for t in tokens if t)

In [3]:
df["genres_tok"] = df["genres"].replace("(no genres listed)", "").apply(lista_para_tokens)
df["director_tok"] = df["director"].apply(lista_para_tokens)
df["cast_tok"] = df["cast"].apply(lambda v: lista_para_tokens(v, limite=3))
df["keywords_tok"] = df["keywords"].apply(lista_para_tokens)

df["meta"] = (
    df["genres_tok"] + " " + df["director_tok"] + " "
    + df["cast_tok"] + " " + df["keywords_tok"]
)
df["sinopse"] = (df["overview"].fillna("") + " " + df["tagline"].fillna("")).str.lower()

df[["title", "meta"]].head()

,title,meta
0,Toy Story (1995),adventure animation children comedy fantasy jo...
1,Jumanji (1995),adventure children fantasy joejohnston robinwi...
2,Grumpier Old Men (1995),comedy romance howarddeutch waltermatthau jack...
3,Waiting to Exhale (1995),comedy drama romance forestwhitaker whitneyhou...
4,Father of the Bride Part II (1995),comedy charlesshyer stevemartin dianekeaton ma...


In [4]:
vec_meta = TfidfVectorizer(token_pattern=r"\S+")
vec_sinopse = TfidfVectorizer(stop_words="english", min_df=2, max_features=20000)

m_meta = vec_meta.fit_transform(df["meta"])
m_sinopse = vec_sinopse.fit_transform(df["sinopse"])

PESO_META = 1.0
PESO_SINOPSE = 0.6

matriz_v2 = hstack([PESO_META * m_meta, PESO_SINOPSE * m_sinopse]).tocsr()
print("Formato:", matriz_v2.shape)

Formato: (9742, 48199)


In [6]:
tags_por_filme = (
    tags.assign(tag=tags["tag"].astype(str).str.lower())
    .groupby("movieId")["tag"]
    .apply(" ".join)
    .reset_index(name="tags")
)

v1 = df[["movieId"]].merge(tags_por_filme, on="movieId", how="left")
texto_v1 = (
    df["genres"].replace("(no genres listed)", "").str.replace("|", " ", regex=False).str.lower()
    + " " + v1["tags"].fillna("")
)
matriz_v1 = TfidfVectorizer(stop_words="english").fit_transform(texto_v1)

In [7]:
indices = pd.Series(df.index, index=df["movieId"])


def buscar_id(trecho):
    achados = df[df["title"].str.contains(trecho, case=False, regex=False)]
    return achados[["movieId", "title"]]


def similares(matriz, movie_id, n=10):
    idx = indices[movie_id]
    scores = cosine_similarity(matriz[idx], matriz).flatten()
    top = scores.argsort()[::-1][1 : n + 1]

    resultado = df.iloc[top][["movieId", "title", "genres"]].copy()
    resultado["similaridade"] = scores[top].round(3)
    return resultado

In [8]:
similares(matriz_v1, 1)

,movieId,title,genres,similaridade
1757,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.862
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.644
8695,122918,Guardians of the Galaxy 2 (2017),Action|Adventure|Sci-Fi,0.368
8219,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,0.358
8927,136016,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy,0.358
3000,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.358
9430,166461,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,0.358
6486,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,0.358
6194,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,0.358
6948,65577,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,0.358


In [9]:
similares(matriz_v2, 1)

,movieId,title,genres,similaridade
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.352
7355,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.347
7639,87876,Cars 2 (2011),Adventure|Animation|Children|Comedy|IMAX,0.166
9274,157296,Finding Dory (2016),Adventure|Animation|Comedy,0.136
9565,173873,Gulliver's Travels (1996),Adventure|Children|Fantasy,0.135
1757,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.133
6212,45517,Cars (2006),Animation|Children|Comedy,0.128
8988,139130,Afro Samurai (2007),Action|Adventure|Animation|Drama|Fantasy,0.120
3680,5069,Escaflowne: The Movie (Escaflowne) (2000),Action|Adventure|Animation|Drama|Fantasy,0.120
9167,148675,North Pole: Open For Christmas (2015),Children|Fantasy,0.116


In [10]:
for nome, m in [("v1", matriz_v1), ("v2", matriz_v2)]:
    r = similares(m, 1, 10)
    print(nome, "- valores distintos no top 10:", r["similaridade"].nunique())

v1 - valores distintos no top 10: 4
v2 - valores distintos no top 10: 9


In [11]:
buscar_id("Dark Knight")

,movieId,title
6710,58559,"Dark Knight, The (2008)"
7768,91529,"Dark Knight Rises, The (2012)"
8032,98124,"Batman: The Dark Knight Returns, Part 1 (2012)"
8080,99813,"Batman: The Dark Knight Returns, Part 2 (2013)"


In [14]:
similares(matriz_v1, 58559)

,movieId,title,genres,similaridade
1060,1377,Batman Returns (1992),Action|Crime,0.507
509,592,Batman (1989),Action|Crime|Thriller,0.483
126,153,Batman Forever (1995),Action|Adventure|Comedy|Crime,0.478
5061,7934,Zelig (1983),Comedy,0.432
3589,4920,"Now, Voyager (1942)",Drama|Romance,0.432
8387,109850,Need for Speed (2014),Action|Crime|Drama|IMAX,0.423
5302,8784,Garden State (2004),Comedy|Drama|Romance,0.419
2044,2723,Mystery Men (1999),Action|Comedy|Fantasy,0.414
5917,33794,Batman Begins (2005),Action|Crime|IMAX,0.408
2897,3877,Supergirl (1984),Action|Adventure|Fantasy,0.399


In [15]:
similares(matriz_v2, 58559)

,movieId,title,genres,similaridade
5917,33794,Batman Begins (2005),Action|Crime|IMAX,0.337
7768,91529,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX,0.329
509,592,Batman (1989),Action|Crime|Thriller,0.231
2418,3213,Batman: Mask of the Phantasm (1993),Animation|Children,0.205
1060,1377,Batman Returns (1992),Action|Crime,0.188
1174,1562,Batman & Robin (1997),Action|Adventure|Fantasy|Thriller,0.185
8963,136864,Batman v Superman: Dawn of Justice (2016),Action|Adventure|Fantasy|Sci-Fi,0.178
7380,79274,Batman: Under the Red Hood (2010),Action|Animation,0.176
126,153,Batman Forever (1995),Action|Adventure|Comedy|Crime,0.162
7455,81564,Megamind (2010),Action|Animation|Children|Comedy|Sci-Fi|IMAX,0.154
